In [ ]:
#!pip install fitter openpyxl
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

In [ ]:
df = pd.read_excel('Concatenation.xlsx')
df.head()

,Station,Bus arrival time,Opening the doors,Closing the doors,Number of entries,Number of exits,Bus movement,position,Date,Time,duration,Waiting Time(s),Waiting Time
0,خاوران,06:40:14,2025-06-02 06:40:18,2025-06-02 06:41:55,22,0,06:41:58,جنوب به شمال,شنبه,صبح,0.001123,97,97
1,بسیج,06:46:17,2025-06-02 06:46:30,2025-06-02 06:46:43,13,0,06:46:43,جنوب به شمال,شنبه,صبح,0.000150,13,13
2,رحمانی,06:47:09,2025-06-02 06:47:16,2025-06-02 06:47:25,12,0,06:47:29,جنوب به شمال,شنبه,صبح,0.000104,9,9
3,ولیعصر,06:48:47,2025-06-02 06:48:49,2025-06-02 06:49:10,1,1,06:49:12,جنوب به شمال,شنبه,صبح,0.000243,21,21
4,رحمانی,07:09:33,2025-06-02 07:09:38,2025-06-02 07:09:48,7,4,07:09:55,جنوب به شمال,شنبه,صبح,0.000116,10,10


In [ ]:
position = ['جنوب به شمال','شمال به جنوب']
Time = ['صبح','ظهر','شب']
Station = ['خاوران','بسیج','رحمانی','ولیعصر']
results = []
dict_df = {}

In [ ]:
def get_position_code(position_name):
    if position_name == 'جنوب به شمال':
        return 'S'
    elif position_name == 'شمال به جنوب':
        return 'N'


In [ ]:
def get_Time_code(Time_name):
    if Time_name == 'صبح':
        return 'M'
    elif Time_name == 'ظهر':
        return 'A'
    elif Time_name == 'شب':
        return 'N'


In [ ]:
def get_Station_code(Station_name):
    if Station_name == 'خاوران':
        return 'K'
    elif Station_name == 'بسیج':
        return 'B'
    elif Station_name == 'رحمانی':
        return 'R'
    elif Station_name == 'ولیعصر':
        return 'V'

In [ ]:
KSM = df[(df['Station'] == 'خاوران') & (df['position'] == 'جنوب به شمال') & (df['Time'] == 'صبح')]
KSM.head()

,Station,Bus arrival time,Opening the doors,Closing the doors,Number of entries,Number of exits,Bus movement,position,Date,Time,duration,Waiting Time(s),Waiting Time
0,خاوران,06:40:14,2025-06-02 06:40:18,2025-06-02 06:41:55,22,0,06:41:58,جنوب به شمال,شنبه,صبح,0.001123,97,97
6,خاوران,07:21:06,2025-06-02 07:21:16,2025-06-02 07:24:02,37,0,07:24:04,جنوب به شمال,شنبه,صبح,0.001921,166,166
55,خاوران,06:50:16,2025-06-02 06:50:21,2025-06-02 06:54:56,25,0,06:55:13,جنوب به شمال,سه شنبه,صبح,0.003183,275,275
59,خاوران,07:18:20,2025-06-02 07:18:24,2025-06-02 07:23:37,39,0,07:23:38,جنوب به شمال,سه شنبه,صبح,0.003623,313,313
62,خاوران,07:43:02,2025-06-02 07:43:04,2025-06-02 07:47:42,39,0,07:47:43,جنوب به شمال,سه شنبه,صبح,0.003218,278,278


In [ ]:
count = 0
for i in Station:
  for j in position :
    for k in Time:
       key = f"{i}_{j}_{k}"
       df_temp = df[(df['Station'] == i) & (df['position'] == j) & (df['Time'] == k)]
       dict_df[key] = {'Name' : get_Station_code(i)+get_position_code(j)+get_Time_code(k),'DF':df_temp}
       count+=1

print(count)


24


In [ ]:
distributions = ['norm', 'expon', 'lognorm', 'gamma', 'beta', 'uniform']

In [ ]:
dict = {
    'name':[],
    'dist_name':[],
    'd_statistic':[],
    'ks_p_value':[],
    'mean':[],
    'variance':[],
    'std':[]
}

In [ ]:
def get_input(Column_Name):
  for i,j in dict_df.items():
    pow = (j['DF'][Column_Name],j['Name'])
    for dist_name in distributions:
      try:
        dist = getattr(stats, dist_name)  # گرفتن شیء توزیع از scipy.stats
        params = dist.fit(pow[0])            # برازش پارامترها به داده‌ها
        D, p = stats.kstest(pow[0], dist_name, args=params)  # انجام آزمون K-S
        dict['name'].append(pow[1])
        dict['dist_name'].append(dist_name)
        dict['d_statistic'].append(D)
        dict['ks_p_value'].append(p)
        dict['mean'].append(pow[0].mean())
        dict['variance'].append(pow[0].var())
        dict['std'].append(pow[0].std())
      except Exception as e:
          continue
    lambda_poisson = np.mean(pow[0])
    data_sorted = np.sort(pow[0])
    ecdf = np.arange(1, len(data_sorted) + 1) / len(data_sorted)
    poisson_cdf = stats.poisson.cdf(data_sorted, lambda_poisson)
    D_poisson = np.max(np.abs(ecdf - poisson_cdf))
    dict['name'].append(pow[1])
    dict['dist_name'].append('poisson')
    dict['d_statistic'].append(D_poisson)
    dict['ks_p_value'].append('-')
    dict['mean'].append(pow[0].mean())
    dict['variance'].append(pow[0].var())
    dict['std'].append(pow[0].std())

 #   plt.hist(pow[0], bins=np.arange(pow[0].min(), pow[0].max()+2)-0.5, density=True, alpha=0.6, color='g', label=f'Data histogram of {pow[1]}')
 #   x = np.arange(pow[0].min(), pow[0].max()+1)
  #  plt.plot(x, stats.poisson.pmf(x, lambda_poisson), '-b', ms=8, label=f'Poisson of {pow[1]}')
  #  plt.xlabel('Value')
  #  plt.ylabel('Probability')
  #  plt.title(f'Data histogram and Poisson for {Column_Name} {pow[1]}')
  #  plt.savefig(f'{pow[1]}.png')
  #  plt.close()





In [ ]:
get_input('Number of entries')

In [ ]:
get_input('Number of exits')

In [ ]:
get_input('Waiting Time')

/usr/local/lib/python3.11/dist-packages/scipy/stats/_continuous_distns.py:795: RuntimeWarning: invalid value encountered in sqrt
  sk = 2*(b-a)*np.sqrt(a + b + 1) / (a + b + 2) / np.sqrt(a*b)
/usr/local/lib/python3.11/dist-packages/scipy/stats/_continuous_distns.py:800: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  a, b = optimize.fsolve(func, (1.0, 1.0))
/usr/local/lib/python3.11/dist-packages/scipy/stats/_continuous_distns.py:6921: RuntimeWarning: overflow encountered in divide
  return np.sum((1 + np.log(shifted/scale)/shape**2)/shifted)


In [ ]:
dataframe1 = pd.DataFrame(dict)

In [ ]:
dataframe1.head()

,name,dist_name,d_statistic,ks_p_value,mean,variance,std
0,KSM,norm,0.215862,0.426818,191.133333,7796.695238,88.298897
1,KSM,expon,0.273400,0.175701,191.133333,7796.695238,88.298897
2,KSM,lognorm,0.183307,0.62999,191.133333,7796.695238,88.298897
3,KSM,gamma,0.177480,0.668589,191.133333,7796.695238,88.298897
4,KSM,beta,0.252986,0.247127,191.133333,7796.695238,88.298897


In [ ]:
dataframe1.to_excel('Info.xlsx',index=False)

In [ ]:
#f = Fitter(entries)
#f.fit()
#f.summary()